In [1]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), 'C\u1ea7n GPU!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb >= 15, f'C\u00e2n \u226515GB VRAM (hi\u1ec7n c\u00f3 {vram_gb:.1f}GB)'

# Check SM version for Flash Attention 2 support
# FA2 requires Ampere+ (SM >= 8.0): A100, A10, 3090, 4090
# T4 (Turing, SM 7.5) does NOT support FA2 - will use SDPA instead
major, minor = torch.cuda.get_device_capability()
HAS_FA2 = major >= 8
print(f'SM version: {major}.{minor}')
print(f'Flash Attention 2: {"supported" if HAS_FA2 else "NOT supported (will use SDPA)"}')


GPU: Tesla T4
VRAM: 15.6 GB
SM version: 7.5
Flash Attention 2: NOT supported (will use SDPA)


In [2]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# Unsloth: kéo sẵn torch, transformers, trl, peft, bitsandbytes, accelerate,
#           flash-attn (wheel prebuilt) với version tương thích
!pip install -q unsloth

# Deps còn lại cho data/eval (không bị Unsloth bundle)
!pip install -q datasets sentencepiece scikit-learn pandas python-dotenv pyyaml huggingface-hub protobuf tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/22

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
HF_REPO = user_secrets.get_secret("HF_REPO")


In [4]:
import os
import re
import json
import glob
import time
import math
from collections import Counter, defaultdict
def load_json_files(dataset_path):
    files = glob.glob(os.path.join(dataset_path, "*.json"))
    print(f"Found {len(files)} JSON files in '{dataset_path}'")
    items = []
    for p in files:
        try:
            with open(p, "r", encoding="utf-8") as f:
                items.append(json.load(f))
        except Exception as e:
            print(f"Error reading {p}: {e}")
    return items
text1= load_json_files("/kaggle/input/tempo-run-2025-run-with-ai-break-limits/Dataset/Dataset/train")
text1[0]


Found 1500 JSON files in '/kaggle/input/tempo-run-2025-run-with-ai-break-limits/Dataset/Dataset/train'


{'url': 'https://thanhnien.vn/sap-khai-mac-trien-lam-nganh-giay-bao-bi-dien-nang-luong-tu-dong-hoa-185240426102412567.htm',
 'title': 'Sắp khai mạc triển lãm ngành giấy, bao bì, điện, năng lượng, tự động hóa',
 'content': 'Từ ngày 8 - 10.5 tại Trung tâm Triển lãm quốc tế WTC Expo tỉnh Bình Dương sẽ diễn ra Triển lãm quốc tế giấy và bao bì Việt Nam - VPPE 2024 và Triển lãm quốc tế ngành điện, năng lượng, máy móc thiết bị công nghiệp, tự động hóa Việt Nam lần 3 - EMA Vietnam 2024.\nNgành giấy là một trong những ngành kinh tế lớn, có tầm quan trọng trong cơ cấu kinh tế nói chung và kinh tế công nghiệp nói riêng. Bên cạnh là ngành phụ trợ quan trọng cho các ngành khác như điện tử, may mặc, da giày, thực phẩm, đồ gỗ…, thì ngành giấy còn cung cấp nhiều sản phẩm cho mục đích đa dạng từ hàng hóa tiêu dùng thiết yếu đến các hoạt động văn hóa, xã hội, giáo dục, y tế, thông tin truyền thông… Trong định hướng phát triển, đến năm 2030, ngành giấy Việt Nam sẽ trở thành ngành sản xuất lớn của khu vực

In [5]:
# 7. Build SFT JSONL — inline tất cả logic
import json, glob, random
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

REPO_DIR = Path('/kaggle/working/')
TRAIN_DIR = Path('/kaggle/input/tempo-run-2025-run-with-ai-break-limits/Dataset/Dataset/train')
OUT_DIR = REPO_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# System Prompt sử dụng cho chiến thuật Chain-of-Thought
SYSTEM_PROMPT = (
    "Bạn là một chuyên gia giải quyết vấn đề xuất sắc. Nhiệm vụ của bạn là đọc kỹ văn bản, "
    "câu hỏi và các lựa chọn, suy luận từng bước dựa trên thông tin trong bài. "
    "Cuối cùng, bắt buộc chốt lại bằng định dạng chính xác: 'Do đó, đáp án đúng là [A/B/C/D]'."
)

def format_choices(choices):
    """Render choices deterministically A-D."""
    ordered = []
    for k in ['A', 'B', 'C', 'D']:
        if k in choices and choices[k] is not None:
            ordered.append(f'{k}: {choices[k]}')
    return '\n'.join(ordered)

def build_user_instruction(title, content, question, choices):
    """Build user-message text sử dụng cấu trúc phân tách rõ ràng."""
    ct = format_choices(choices)
    
    # Sử dụng Markdown để khoanh vùng dữ liệu
    return (
        f'### VĂN BẢN\n'
        f'**Tiêu đề:** {title}\n'
        f'**Nội dung:**\n{content}\n\n'
        f'### CÂU HỎI\n{question}\n\n'
        f'### LỰA CHỌN\n{ct}\n\n'
        f'Vui lòng cung cấp đáp án của bạn (A, B, C, hoặc D):'
    )

def build_messages(system_prompt, user_instruction, assistant=None):
    msgs = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_instruction},
    ]
    if assistant is not None:
        msgs.append({'role': 'assistant', 'content': assistant})
    return msgs

# Load all JSON files and build rows
rows = []
dropped = 0
for path in sorted(glob.glob(str(TRAIN_DIR / '*.json'))):
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    
    content = (d.get('content:') or d.get('content') or '').strip()
    title = (d.get('title:') or d.get('title') or '').strip()
    questions = d.get('questions') or []
    
    if not content or not questions:
        dropped += 1
        continue
        
    for idx, q in enumerate(questions, start=1):
        question = (q.get('question') or '').strip()
        choices = q.get('choices') or {}
        raw_label = q.get('correct_answer')
        explanation = (q.get('explanation') or '').strip()
        label = raw_label.strip().upper()[:1] if raw_label else None
        
        if not question or not choices:
            continue
            
        # CHẶN ĐỨNG các câu không có label hoặc label không hợp lệ
        if label is None or label not in {'A', 'B', 'C', 'D'}:
            continue
            
        user = build_user_instruction(title, content, question, choices)
        
        # Buid câu trả lời CoT cho Assistant
        if explanation:
            assistant_reply = f"{explanation}\n\nDo đó, đáp án đúng là {label}."
        else:
            assistant_reply = f"Do đó, đáp án đúng là {label}."
            
        msgs = build_messages(SYSTEM_PROMPT, user, assistant=assistant_reply)
        
        row = {
            'messages': msgs,
            'title': title, 
            'content': content,
            'question': question, 
            'choices': choices,
            'explanation': explanation,
            'label': label
        }
        rows.append(row)

print(f'Built {len(rows)} rows (dropped {dropped} docs)')

# Stratified split 90/10 với list đã sạch 100% label
labels = [r['label'] for r in rows]
train_rows, eval_rows = train_test_split(
    rows, test_size=0.1, random_state=3407, shuffle=True, stratify=labels,
)

print(f'Train: {len(train_rows)}  Eval: {len(eval_rows)}')
print(f'Train label dist: {dict(Counter(r["label"] for r in train_rows))}')
print(f'Eval  label dist: {dict(Counter(r["label"] for r in eval_rows))}')

def write_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

write_jsonl(train_rows, OUT_DIR / 'train.jsonl')
write_jsonl(eval_rows, OUT_DIR / 'eval.jsonl')
print(f'Wrote {OUT_DIR / "train.jsonl"} and {OUT_DIR / "eval.jsonl"}')

Built 4491 rows (dropped 3 docs)
Train: 4041  Eval: 450
Train label dist: {'B': 1638, 'A': 1257, 'D': 144, 'C': 1002}
Eval  label dist: {'C': 112, 'B': 182, 'A': 140, 'D': 16}
Wrote /kaggle/working/data/processed/train.jsonl and /kaggle/working/data/processed/eval.jsonl


In [6]:
# 8. Balance labels — inline balance_via_reorder + merge
# Reuse: build_user_instruction, build_messages, write_jsonl, read_jsonl,
#         train_test_split from cell 7
from collections import Counter
import random

PROC_DIR = REPO_DIR / 'data' / 'processed'
FINAL_DIR = PROC_DIR / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

def reorder_row_for_label(row, target_label):
    """Rotate choices sao cho correct answer lands on target_label."""
    orig_label = row.get('label', '')
    if orig_label not in {'A','B','C','D'} or target_label not in {'A','B','C','D'}:
        return row
        
    choices = row.get('choices') or {}
    if not all(k in choices for k in ('A','B','C','D')):
        return row
        
    shift = (ord(target_label) - ord(orig_label)) % 4
    if shift == 0:
        return row
        
    new_choices = {}
    for k, v in choices.items():
        if k in {'A','B','C','D'}:
            new_choices[chr(ord('A') + (ord(k) - ord('A') + shift) % 4)] = v
        else:
            new_choices[k] = v
            
    new_user = build_user_instruction(
        row.get('title',''), row.get('content',''),
        row.get('question',''), new_choices,
    )
    
    # --- PHẦN FIX LỖI Ở ĐÂY ---
    explanation = row.get('explanation', '').strip()
    if explanation:
        new_assistant_reply = f"{explanation}\n\nDo đó, đáp án đúng là {target_label}."
    else:
        new_assistant_reply = f"Do đó, đáp án đúng là {target_label}."
        
    new_msgs = build_messages(SYSTEM_PROMPT, new_user, assistant=new_assistant_reply)
    # --------------------------
    
    new_row = dict(row)
    new_row['messages'] = new_msgs
    new_row['choices'] = new_choices
    new_row['label'] = target_label
    
    return new_row

def balance_via_reorder(rows, seed=3407):
    """Rebalance by rotating choices so label cycles A->B->C->D."""
    rng = random.Random(seed)
    indices = list(range(len(rows)))
    rng.shuffle(indices)
    target_labels = ['A', 'B', 'C', 'D']
    return [reorder_row_for_label(rows[i], target_labels[i % 4]) for i in indices]

# 1. Load train và balance nó
train_rows = read_jsonl(PROC_DIR / 'train.jsonl')
pre_dist = dict(Counter(r['label'] for r in train_rows)) # Đã sạch UNK nhờ Cell 7
print(f'Train Before balance: {pre_dist}')

final_train = balance_via_reorder(train_rows, seed=3407)
post_dist = dict(Counter(r['label'] for r in final_train))
print(f'Train After balance:  {post_dist}')

# 2. Load eval (Giữ nguyên, không động chạm gì vào data này)
final_eval = read_jsonl(PROC_DIR / 'eval.jsonl')
eval_dist = dict(Counter(r['label'] for r in final_eval))
print(f'Eval dist (Original): {eval_dist}')

print(f'Total: {len(final_train) + len(final_eval)} (Train {len(final_train)} + Eval {len(final_eval)})')

# 3. Ghi ra file luôn
write_jsonl(final_train, FINAL_DIR / 'train.jsonl')
write_jsonl(final_eval, FINAL_DIR / 'eval.jsonl')
print(f'Wrote {FINAL_DIR}')


Train Before balance: {'B': 1638, 'A': 1257, 'D': 144, 'C': 1002}
Train After balance:  {'A': 1011, 'D': 1010, 'C': 1010, 'B': 1010}
Eval dist (Original): {'C': 112, 'B': 182, 'A': 140, 'D': 16}
Total: 4491 (Train 4041 + Eval 450)
Wrote /kaggle/working/data/processed/final


In [7]:
# 9. Load Unsloth model + LoRA
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen3-0.6B'
MAX_SEQ_LENGTH = 4096
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# Load model + tokenizer — Unsloth tự xử lý FA2 + Triton kernels
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto: bf16 trên A100, fp16 trên cũ hơn
    load_in_4bit=True,
)
print(f'Loaded {MODEL_NAME}, vocab={tokenizer.vocab_size}')

# Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)
model.print_trainable_parameters()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/576M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loaded Qwen/Qwen3-0.6B, vocab=151643


Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 20,185,088 || all params: 616,235,008 || trainable%: 3.2756


In [8]:
# 10. Render chat text — cách tự nhiên (không dùng thẻ think)
# Reuse: train_rows from cell 7

sample = final_train[0]  # từ cell 8
msgs = sample['messages']
print('=== Messages ===')
for m in msgs:
    print(f'  [{m["role"]}] {m["content"][:80]}...')

# Hàm render mới: Cực kỳ ngắn gọn
def render_chat_for_training(tokenizer, messages):
    """
    Render toàn bộ messages thành 1 string duy nhất cho SFT.
    Không cần cắt ghép thủ công nữa.
    """
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False, # Đặt False vì messages ĐÃ CÓ role assistant ở cuối rồi
        enable_thinking=False        # Khóa chặn hoàn toàn mọi thẻ <think> tự động
    )
    return text

train_text = render_chat_for_training(tokenizer, msgs)

print('\n=== Training text (Tự nhiên) ===')
print(repr(train_text[-120:]))

=== Messages ===
  [system] Bạn là một chuyên gia giải quyết vấn đề xuất sắc. Nhiệm vụ của bạn là đọc kỹ văn...
  [user] ### VĂN BẢN
**Tiêu đề:** Góc khuất đời nghệ sĩ
**Nội dung:**
Góc khuất đời nghệ ...
  [assistant] Bài viết nêu rõ: 'Sau đợt dịch, không có việc, Hiệp Vịt lên Facebook thử sang lĩ...

=== Training text (Tự nhiên) ===
'hà hát, không phải nguồn thu nhập bổ sung đáng kể sau dịch COVID-19 như hát bolero.\n\nDo đó, đáp án đúng là A.<|im_end|>\n'


In [9]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import TrainerCallback
import torch
# Reuse: render_chat_for_training, read_jsonl from cell 10 / cell 7

TRAIN_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'train.jsonl'
EVAL_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'eval.jsonl'
OUTPUT_DIR = REPO_DIR / 'artifacts' / 'unsloth_qwen3_0_6b'

train_rows = read_jsonl(TRAIN_JSONL)
eval_rows = read_jsonl(EVAL_JSONL)
print(f'Train: {len(train_rows)}  Eval: {len(eval_rows)}')



# Render to text with enable_thinking
def to_text(row):
    return {'text': render_chat_for_training(tokenizer, row['messages'])}

train_ds = Dataset.from_list(train_rows).map(
    to_text, remove_columns=list(train_rows[0].keys()),
)
eval_ds = Dataset.from_list(eval_rows).map(
    to_text, remove_columns=list(eval_rows[0].keys()),
)
print(f'Dataset: train={len(train_ds)} eval={len(eval_ds)}')
print(f'Sample text (last 100 chars): {repr(train_ds[0]["text"][-100:])}')


# --- BẮT ĐẦU PHẦN FIX ---

# 1. Khai báo Data Collator để chỉ tính loss phần assistant
# response_template = "<|im_start|>assistant\n"
# collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# 2. SFT Config chuẩn
sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=4,
    gradient_accumulation_steps=4,  # effective batch = 32
    num_train_epochs=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type='cosine',
    logging_steps=20,
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    max_grad_norm=0.3,
    optim='adamw_8bit',
    torch_empty_cache_steps=100,
    
    # FIX: Tự động chạy fp16 trên T4, bf16 trên A100/H100
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    gradient_checkpointing=True,
    dataset_text_field='text',
    packing=False,
    max_seq_length=MAX_SEQ_LENGTH, # FIX: trl bản mới dùng max_seq_length
    eval_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=[],
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

# Kích hoạt tính năng chỉ tính loss trên câu trả lời (completion_only_loss)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",    # Dấu hiệu nhận biết câu hỏi
    response_part="<|im_start|>assistant\n", # Dấu hiệu nhận biết câu trả lời
)


# --- KẾT THÚC PHẦN FIX ---


print('Starting training...')
trainer.train()

# Save LoRA adapter
adapter_dir = OUTPUT_DIR / 'adapter'
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f'Saved adapter to {adapter_dir}')

# Save merged 16-bit (optional, cho standalone inference)
merged_dir = OUTPUT_DIR / 'merged_16bit'
merged_dir.mkdir(parents=True, exist_ok=True)
try:
    model.save_pretrained_merged(str(merged_dir), tokenizer, save_method='merged_16bit')
    print(f'Saved merged 16-bit to {merged_dir}')
except Exception as e:
    print(f'save_pretrained_merged failed: {e}')

Train: 4041  Eval: 450


Map:   0%|          | 0/4041 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Dataset: train=4041 eval=450
Sample text (last 100 chars): 'guồn thu nhập bổ sung đáng kể sau dịch COVID-19 như hát bolero.\n\nDo đó, đáp án đúng là A.<|im_end|>\n'


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/4041 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:04<00:00,  4.78s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:07<00:00,  7.10s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/artifacts/unsloth_qwen3_0_6b/merged_16bit`
Saved merged 16-bit to /kaggle/working/artifacts/unsloth_qwen3_0_6b/merged_16bit


In [10]:
import gc
import torch

# 1. Xóa biến trainer nếu nó vẫn còn tồn tại trong RAM
if 'trainer' in globals():
    del trainer

# 2. Xóa sạch bộ nhớ đệm của GPU
torch.cuda.empty_cache()

# 3. Ép Python dọn rác hệ thống
gc.collect()

print("Đã dọn dẹp VRAM! Sẵn sàng chạy Eval.")

Đã dọn dẹp VRAM! Sẵn sàng chạy Eval.


In [11]:
# 13. Generate submission
import json, glob
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Reuse: to_chat_prompt, letter_ids, model, tokenizer, read_jsonl from cell 12
#         build_user_instruction from cell 7

TEST_DIR = Path('/kaggle/input/datasets/phcthnho/tempo-2025/Dataset/Dataset/test')
OUT_CSV = REPO_DIR / 'submissions' / 'sub_unsloth.csv'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
MAX_LENGTH = 2048
BATCH_SIZE = 4
SYSTEM_PROMPT = 'Bạn là hệ thống trả lời trắc nghiệm. Chỉ xuất duy nhất 1 ký tự A/B/C/D.'

# Walk test JSON files — same logic as infer.py
items = []
for path in sorted(glob.glob(str(TEST_DIR / '*.json'))):
    article_id = Path(path).stem
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    content = (d.get('content:') or d.get('content') or '').strip()
    title = (d.get('title:') or d.get('title') or '').strip()
    questions = d.get('questions') or []
    if not content or not questions:
        continue
    for idx, q in enumerate(questions, start=1):
        question = (q.get('question') or '').strip()
        choices = q.get('choices') or {}
        if not question or not choices:
            continue
        items.append({
            'article_id': article_id,
            'qid': idx,
            'row_id': f'{article_id}__q{idx}',
            'title': title,
            'content': content,
            'question': question,
            'choices': choices,
        })

print(f'Test items: {len(items)}')

# Build prompts
prompts = []
for it in items:
    instr = build_user_instruction(it['title'], it['content'], it['question'], it['choices'])
    prompts.append(to_chat_prompt(tokenizer, instr, SYSTEM_PROMPT))

# Predict — same logits mode as eval
results = []
with torch.no_grad():
    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc='Inferring'):
        ps = prompts[i:i+BATCH_SIZE]
        enc = tokenizer(ps, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(model.device)
        
        # 1. Bring back the memory optimization to prevent OOM
        outputs = model(**enc, num_logits_to_keep=1)
        
        # 2. Because we only kept 1 logit, the sequence dimension is 1.
        # We don't need any complex attention_mask math. Just grab index 0!
        next_logits = outputs.logits[:, 0, :]
        
        cand_ids = torch.tensor(list(letter_ids.values()), device=outputs.logits.device)
        cand_logits = next_logits[:, cand_ids]
        
        pred_idx = cand_logits.argmax(dim=1)
        idx2letter = list(letter_ids.keys())
        
        for j, p in enumerate(pred_idx):
            results.append({
                'row_id': items[i + j]['row_id'],
                'answer': idx2letter[p.item()],
            })
        del enc, outputs, next_logits, cand_logits
        torch.cuda.empty_cache()

df = pd.DataFrame(results)
df = df[['row_id', 'answer']]
df.to_csv(OUT_CSV, index=False)
print(f'\nWrote {len(df)} rows to {OUT_CSV}')
print(f'Pred dist: {df["answer"].value_counts().to_dict()}')
print(f'Duplicates: {df["row_id"].duplicated().sum()}')
print(df.head(10))


Test items: 1488


NameError: name 'to_chat_prompt' is not defined